# Graph Deep learning model
In the other tutorials, all examples have used scikit-learn models. However,
QSPRpred also has a number of other deep-learning models build-in. These models rely on
torch, therefore you need to make sure to have torch or installed QSPPred with the `deep` (or `full`) option (see [README.txt](https://github.com/CDDLeiden/QSPRpred#readme)). 

This tutorial is heavily inspired by the *deep_learning_models* tutorial.

First, we will load the dataset as usual.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import importlib
from qsprpred.data import QSPRDataset, RandomSplit
#from qsprpred.data.descriptors.fingerprints import MorganFP
from qsprpred.data.descriptors.sets import SmilesDesc

import sys
sys.path.insert(0, '/home/ubuntu/implementation/QSPRpred')

#import qsprpred.extra.gpu.models.gdnn as gdnn_module
#from importlib import reload
#
#reload(gdnn_module)

modname = 'qsprpred.extra.gpu.models.gdnn'
if modname in sys.modules:
    del sys.modules[modname]

import qsprpred.extra.gpu.models.gdnn as gdnn_module
from qsprpred.extra.gpu.models.gdnn import GGNN
importlib.reload(gdnn_module)

#from qsprpred.extra.gpu.models.gdnn import DNNModel, GGNN
#print(DNNModel.__init__.__code__.co_varnames)



<module 'qsprpred.extra.gpu.models.gdnn' from '/home/ubuntu/implementation/QSPRpred/qsprpred/extra/gpu/models/gdnn.py'>

In [2]:


os.makedirs("../../tutorial_output/data", exist_ok=True)

# Create dataset

# setting task as Classification, treshold is therefore crucial ("th") - has to be provided as a list of floats
dataset = QSPRDataset.fromTableFile(
    filename="../../tutorial_data/A2A_LIGANDS.tsv",
    store_dir="../../tutorial_output/data",
    name="DeepLearningTutorialDataset",
    target_props=[{"name": "pchembl_value_Mean", "task": "SINGLECLASS", "th": [6.5]}],
    random_state=42
)

dataset.prepareDataset(
    split=RandomSplit(test_fraction=0.2, dataset=dataset),
    feature_calculators=[SmilesDesc()],
    recalculate_features=False,
)

dataset.getDF().head()


#print(DNNModel.__init__.__code__.co_varnames)

,SMILES,pchembl_value_Mean,Year,QSPRID,pchembl_value_Mean_original
QSPRID,,,,,
DeepLearningTutorialDataset_0000,Cc1cc(C)n(-c2cc(NC(=O)CCN(C)C)nc(-c3ccc(C)o3)n...,True,2008.0,DeepLearningTutorialDataset_0000,8.68
DeepLearningTutorialDataset_0001,Nc1c(C(=O)Nc2ccc([N+](=O)[O-])cc2)sc2nc3c(cc12...,False,2010.0,DeepLearningTutorialDataset_0001,4.82
DeepLearningTutorialDataset_0002,O=C(Nc1nc2ncccc2n2c(=O)n(-c3ccccc3)nc12)c1ccccc1,False,2009.0,DeepLearningTutorialDataset_0002,5.65
DeepLearningTutorialDataset_0003,CNC(=O)C12CC1C(n1cnc3c(NCc4cccc(Cl)c4)nc(C#CCC...,False,2009.0,DeepLearningTutorialDataset_0003,5.45
DeepLearningTutorialDataset_0004,CCCn1c(=O)c2c(nc3cc(OC)ccn32)n(CCCNC(=O)c2ccc(...,False,2019.0,DeepLearningTutorialDataset_0004,5.20


## Fully connected neural network
### Initialization
The first model we will look at is a fully connected neural network. This model uses the `DDNModel` class instead of the `SklearnModel` class. The `DDNModel` class accepts a `patience` argument, which is the number of epochs to wait before stopping training if the validation loss does not improve and a tolerance ( `tol`) argument, which is the minimum improvement in validation loss to be considered an improvement.

Other parameters for the underlying estimator `STFullyConnected` can be passed to the `parameters` argument as usual.
There is no need to specify the `alg` argument, as currently only `STFullyConnected` is available.

In [3]:
# Create model

#reload(sys.modules['qsprpred.extra.gpu.models.gdnn'])

#reload(nn_mod)
#print(GGNN.__init__.__code__.co_varnames)

os.makedirs("../../tutorial_output/models", exist_ok=True)
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = gdnn_module.DNNModel(
    base_dir='../../tutorial_output/graph_models',
    name='GraphDeepLearningTutorialModel',
    parameters={'n_epochs': 100,
                'n_dim': 74,          # 74-256 for example
                'patience': 5,
                'in_feats': 74,
                'n_steps': 3,
                'n_etypes': 1,
                'bias': True,
                'n_hidden_layers': 2,
                'dropout_rate': 0.2
               },
    tol=0.01,
    random_state=42
    #device=device
)


print("GGNN init signature:",model.alg.__init__.__code__.co_varnames)

/home/ubuntu/implementation/QSPRpred/qsprpred/extra/gpu/models/gdnn.py:578: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  estimator.load_state_dict(torch.load(path))
qsprpre

GGNN updated
GGNN updated
GGNN updated
GGNN init signature: ('self', 'n_dim', 'device', 'gpus', 'is_reg', 'patience', 'tol', 'parameters', 'n_class', 'i', 'set_in_feats', 'layer', 'pooling_gate_nn')


### Early Stopping
The `DNNModel` supports early stopping of the training, which as mentioned can be controlled by the `patience` and the `tol` arguments. You can check if a model supports early stopping, by checking the `supportsEarlyStopping` attribute.

### Early Stopping
The `DNNModel` supports early stopping of the training, which as mentioned can be controlled by the `patience` and the `tol` arguments. You can check if a model supports early stopping, by checking the `supportsEarlyStopping` attribute.

In [4]:
model.supportsEarlyStopping

True

The model can be trained as usual, but a part of the training set will be used as a validation set to determine wether to stop training. By default a random 10% of the training set is used as a validation set. This can be changed by setting the `split` argument of `QSPRModel.fit` to a different value, which can be any scikit-learn or QSPRpred `DataSplit`. See the [data splitting tutorial](../../basics/data/data_splitting.ipynb) for more information on the possibilities. Here, you can see how to change the validation split to a 20% random split.

In [5]:
from qsprpred.models import CrossValAssessor

CrossValAssessor('r2')(model, dataset,
                       split=RandomSplit(test_fraction=0.2, dataset=dataset))

#X = dataset.getDF()['SMILES']
#y = dataset.getDF()['pchembl_value_Mean_original']
#
#print(X.head())
#
#model.fit(X, y, split=RandomSplit(test_fraction=0.2, dataset=dataset))

#gnn = model.fit(dataset.X, dataset.y, model.alg, split=RandomSplit(test_fraction=0.2, dataset=dataset))

GGNN updated
Fitting...


/home/ubuntu/implementation/QSPRpred/qsprpred/extra/gpu/models/gdnn.py:196: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647352509/work/torch/csrc/utils/tensor_new.cpp:278.)
  labels = torch.tensor(labels).unsqueeze(1).squeeze()#.to(self.device)


GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...


array([-0.96386104, -0.85946399, -1.69834711, -0.62842893, -1.48620199])

If a model supports early stopping, it also has an `EarlyStopping` attribute which is an instance of the `EarlyStopping` class that keeps track of the number of epochs trained. The `EarlyStopping` class has a `mode` attribute that sets how early stopping should be handled when fitting the estimator. It can be one of four modes: `EarlyStoppingMode.RECORDING`, `EarlyStoppingMode.NOT_RECORDING`, `EarlyStoppingMode.OPTIMAL`, `EarlyStoppingMode.FIXED`. You can find [a schematic overview](####early-stopping-modes-overview) of the different modes below.
By default it is set to `EarlyStoppingMode.NOT_RECORDING`.

In [6]:
model.earlyStopping.mode

<EarlyStoppingMode.OPTIMAL: 'OPTIMAL'>

In this mode (`EarlyStoppingMode.NOT_RECORDING`), the `EarlyStopping` class will not keep track of at which epoch the training is stopped in a fit. In the `EarlyStoppingMode.RECORDING` mode the `EarlyStopping` class will keep track of the epoch on which the training was stopped. This can be accessed through the `EarlyStopping` class `trainedEpochs` attribute, which is a list of the epochs on which the training was stopped. You can see that for now it is just an empty list.

In [7]:
model.earlyStopping.trainedEpochs

[1, 1, 1, 1, 1, 1]

If we then run a cross-validation with the mode set to `EarlyStoppingMode.RECORDING`, we can see that the `trainedEpochs` attribute is now filled with the epochs on which the training was stopped for each fold.

In [8]:
from qsprpred.models import EarlyStoppingMode

CrossValAssessor('r2', mode=EarlyStoppingMode.RECORDING)(model, dataset)
model.earlyStopping.trainedEpochs

GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...


[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Mind that the mode has now been changed to `EarlyStoppingMode.RECORDING`, therefore, if you run the cross-validation again, the `trainedEpochs` attribute will be appended with the epochs on which the training was stopped previously.

In [9]:
model.earlyStopping.mode

<EarlyStoppingMode.RECORDING: 'RECORDING'>

In [10]:
#CrossValAssessor('r2')(model, dataset)
model.earlyStopping.trainedEpochs

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

The `EarlyStopping` class has an attribute `optimalEpochs`, which is an aggregation of the `trainedEpochs` attribute, calculated by the `aggregateFunc`, which by default is the arithmetric mean. You can change the `aggregateFunc` by passing a function to the `aggregateFunc` argument of the `EarlyStopping` class. For example, if you want to use the median instead of the mean, you can pass the `np.median` function to the `aggregateFunc` argument.

In [11]:
model.earlyStopping.aggregateFunc

<function mean at 0x7fe4c03ab770>

In [12]:
model.earlyStopping.optimalEpochs

1

In [13]:
import numpy as np
#
model.earlyStopping.aggregateFunc = np.median
model.earlyStopping.optimalEpochs

1

If we now fit the model with the `EarlyStoppingMode.OPTIMAL` mode, the estimator will be fitted for exactly `optimalEpochs`. This is the `EarlyStoppingMode` that is used by default for `QSPRModel.fitDataset`. This is done to avoid having to use a part of the dataset as validation set to determine the early stopping epoch in the final fit of the model. However, if you want to use a different mode, you can pass it to the `mode` argument of the `QSPRModel.fitDataset` method.

In [14]:
_ = model.fitDataset(dataset)

GGNN updated
Fitting...


We can also reset the recorded epochs with the `clean` function. If we then try to run fit attached with the `EarlyStoppingMode.OPTIMAL` mode, we get an error, because the `trainedEpochs` attribute is empty.

In [15]:
model.earlyStopping.clean()
print(model.earlyStopping.trainedEpochs)
try:
    model.fitDataset(dataset)
except ValueError as e:
    print(e)

[]
GGNN updated
No number of epochs have been recorded yet, first run fit with early stopping mode set to RECORDING or set the optimal number of epochs manually.


Another option is to forgo early stopping and just train for a fixed number of epochs. This can be done by setting the `mode` argument to `EarlyStoppingMode.FIXED` and `QSPRModel.earlyStopping.numEpochs` argument to the number of epochs to train for.
Be aware, that if you `DNNModel` parameter `n_epochs` is set to a value smaller than `QSPRModel.earlyStopping.numEpochs`, the model will only be trained for `n_epochs` epochs, as this is the maximum number of epochs to train for.
In this example, we will train the model for 10 epochs.

In [16]:
model.earlyStopping.numEpochs = 10
_ = model.fitDataset(dataset, mode=EarlyStoppingMode.FIXED)

GGNN updated
Fitting...


#### Early stopping modes overview
![EarlyStopping.png](../../figures/EarlyStopping.png)

### Training
Below we will show a complete training of the DNN. First we run hyperparameter optimization to find the best parameters for the model. Here we will will use `EarlyStoppingMode.NOT_RECORDING` as the best epoch to stop training may depend on the hyper-parameters. Then we will apply cross-validation and test set evaluation to get an estimate of the performance of the model, with early stopping set to `EarlyStoppingMode.RECORDING`. Finally, we will use `QSPRModel.fitDataset` training for exactly the average number of epochs trained for in the cross-validation and the test set evaluation.

In [17]:
from qsprpred.models import GridSearchOptimization, TestSetAssessor

# Define the search space
#search_space = {"lr": [1e-4, 1e-3, ], "neurons_h1": [100, 200]}
search_space = {
    "lr": [1e-4, 1e-3, ], 
    "n_hidden_layers": [1, 2],
    "dropout_rate": [0.1, 0.3],
    "in_feats": [74],
    "n_steps": [2, 3],
    "n_etypes": [1],
    "n_dim": [74, 100]
}

gridsearcher = GridSearchOptimization(
    param_grid=search_space,
    model_assessor=TestSetAssessor(
        scoring='r2',
        mode=EarlyStoppingMode.NOT_RECORDING
    ),
)
gridsearcher.optimize(model, dataset)

# Create a CrossValAssessor object
CrossValAssessor('r2', mode=EarlyStoppingMode.RECORDING)(model, dataset)
TestSetAssessor('r2', mode=EarlyStoppingMode.RECORDING)(model, dataset)

model.earlyStopping.aggregateFunc = np.mean
print(
    model.earlyStopping.trainedEpochs)  # list of 6 values, one for each cross-validation fold and one for the test set
print(model.earlyStopping.optimalEpochs)  # average of the 6 values

# fit the model on the whole dataset
_ = model.fitDataset(dataset, mode=EarlyStoppingMode.OPTIMAL)

GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
GGNN updated
Fitting...
[1, 1, 1, 1, 1, 1]
1
GGNN updated
Fitting...
